# 03.0 — Modeling Contract & Experimental Protocol

## 1. Mục tiêu
Notebook này **KHÔNG huấn luyện bất kỳ mô hình nào**. 

Nhiệm vụ duy nhất của notebook này là:
1. Thiết lập và công bố hợp đồng thực nghiệm (**Experimental Protocol**) chuẩn hóa duy nhất cho toàn bộ các notebook modeling downstream.
2. Định nghĩa các quy tắc ranh giới bất di bất dịch (**Zero-Tolerance P0 Rules**):
   - **P0.1 — Direct Multi-Horizon Leakage**: Loại trừ hoàn toàn việc target của cửa sổ $H=7$ vượt qua ranh giới phân tách (Purged / Gap Boundary).
   - **P0.2 — Không chọn threshold / HPO trên Test**: Toàn bộ tinh chỉnh siêu tham số và ngưỡng phân loại $\tau_c$ phải chọn trên tập **Validation**, đóng băng (**Freeze**) tuyệt đối trước khi sang Test.
   - **P0.3 — Chính sách Future Covariates**: Chỉ sử dụng biến khí tượng lịch sử ($t, t-1, \dots$) và biến lịch tương lai đã biết trước (calendar cyclical). Tuyệt đối cấm dùng biến khí tượng tương lai thực tế trong operational forecasting.
   - **P0.4 — Làm rõ chu kỳ thời gian**: Phân biệt rành mạch chu kỳ tuần $s=7$ ngày với chế độ mùa mưa vĩ mô hàng năm (tháng 5 – tháng 11).
   - **P0.5 — Freeze trước Test**: Mô hình được lựa chọn, đóng băng trọng số rồi mới suy diễn trên Test.
3. Xuất file hợp đồng máy đọc được: `data/processed/modeling_contract.json`.


In [1]:
import sys
from pathlib import Path
import json
import time
import warnings
warnings.filterwarnings('ignore')

# Tự động tìm project root từ bất kỳ thư mục con nào
ROOT_DIR = Path.cwd()
while not (ROOT_DIR / "src").exists() and ROOT_DIR != ROOT_DIR.parent:
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.training.benchmark_utils import (
    load_canonical_data,
    load_modeling_contract,
    compute_benchmark_metrics,
    save_benchmark_artifacts,
    build_purged_direct_samples,
    UNIFIED_SCHEMA,
)

from src.training.benchmark_utils import get_default_contract, save_modeling_contract
contract = get_default_contract()
print("Modeling Contract loaded successfully.")


Modeling Contract loaded successfully.


## 2. Xác thực Phân chia Canonical Split & Validation Split

- **Train Full**: `2000-01-01` đến `2020-04-30` (7,426 ngày)
  - **Train Sub**: `2000-01-01` đến `2017-12-31` (Huấn luyện sơ bộ)
  - **Validation**: `2018-01-01` đến `2020-04-30` (Dùng để HPO, chọn threshold $\tau_c$, early stopping)
- **Test Set**: `2020-05-01` đến `2026-04-30` (Đánh giá khách quan cuối cùng, cấm can thiệp)


In [2]:
train_full, val_set, test_set = load_canonical_data()

print(f"Train Full range : {train_full['Ngày'].min().date()} --> {train_full['Ngày'].max().date()} ({len(train_full):,} rows)")
print(f"Validation range : {val_set['Ngày'].min().date()} --> {val_set['Ngày'].max().date()} ({len(val_set):,} rows)")
print(f"Test Set range   : {test_set['Ngày'].min().date()} --> {test_set['Ngày'].max().date()} ({len(test_set):,} rows)")

assert test_set['Ngày'].min().strftime('%Y-%m-%d') == contract['canonical_split_date'], "Test split date mismatch!"
assert val_set['Ngày'].min().strftime('%Y-%m-%d') == contract['validation_split_date'], "Val split date mismatch!"
print(">> Canonical Split Validation: PASS")


Train Full range : 2000-01-01 --> 2020-04-30 (7,426 rows)
Validation range : 2018-01-01 --> 2020-04-30 (851 rows)
Test Set range   : 2020-05-01 --> 2026-04-30 (2,191 rows)
>> Canonical Split Validation: PASS


## 3. Quy chuẩn Metrics: Point-wise & Cumulative Rainfall

Metric dự báo lượng mưa tích lũy trên cửa sổ $H$:
$$A_i(H) = \sum_{h=1}^H y_{i,h}, \quad F_i(H) = \sum_{h=1}^H \hat{y}_{i,h}$$

$$\mathbf{CR\text{-}MAE}@H = \frac{1}{N} \sum_{i=1}^N \left| F_i(H) - A_i(H) \right|$$

$$\mathbf{CR\text{-}Bias}@H = \frac{1}{N} \sum_{i=1}^N \left( F_i(H) - A_i(H) \right)$$

Được đo đạc chi tiết tại các horizon $H \in \{1, 3, 5, 7\}$ ngày, kết hợp cùng các metric point-wise truyền thống: MAE, RMSE, rain_day_MAE, rain_day_RMSE.


In [3]:
print("Standard Evaluation Metrics Schema:")
for idx, col in enumerate(UNIFIED_SCHEMA, 1):
    print(f"  {idx:02d}. {col}")


Standard Evaluation Metrics Schema:
  01. model
  02. group
  03. strategy
  04. MAE
  05. RMSE
  06. rain_day_MAE
  07. rain_day_RMSE
  08. CR-MAE@1
  09. CR-MAE@3
  10. CR-MAE@5
  11. CR-MAE@7
  12. CR-Bias@7
  13. training_time
  14. inference_time
  15. parameter_count


## 4. Xuất Hợp đồng Khoa học (Contract JSON)

In [4]:
out_contract_path = ROOT_DIR / "data" / "processed" / "modeling_contract.json"
save_modeling_contract(contract, out_contract_path)
print(f">> Saved modeling contract to: {out_contract_path}")

with open(out_contract_path, "r", encoding="utf-8") as f:
    saved_c = json.load(f)
print(json.dumps(saved_c, indent=2, ensure_ascii=False))


>> Saved modeling contract to: D:\DS108_project\data\processed\modeling_contract.json
{
  "canonical_split_date": "2020-05-01",
  "validation_split_date": "2018-01-01",
  "target_col": "Lượng mưa",
  "date_col": "Ngày",
  "rain_threshold": 0.1,
  "forecast_horizon": 7,
  "context_length": 30,
  "season_length": 7,
  "annual_macro_regime": "Wet (May-Nov) vs Dry (Dec-Apr)",
  "purge_boundary_policy": "Purged boundary: Origin t must satisfy t + H <= T_train",
  "future_covariate_policy": "Historical meteo lags + calendar features only (no future meteo)",
  "threshold_freeze_policy": "Tuned on validation, frozen strictly before test",
  "schema": [
    "model",
    "group",
    "strategy",
    "MAE",
    "RMSE",
    "rain_day_MAE",
    "rain_day_RMSE",
    "CR-MAE@1",
    "CR-MAE@3",
    "CR-MAE@5",
    "CR-MAE@7",
    "CR-Bias@7",
    "training_time",
    "inference_time",
    "parameter_count"
  ]
}
